In [1]:
import pandas as pd

In [2]:
return_average = pd.read_csv("0.003_benchmark.csv", header = 0, index_col = 0)

In [4]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac

def newey_west_tstat(returns, maxlags=1):
    """
    논문 방식에 따른 Newey-West t-통계량 계산 함수
    입력:
        returns: 수익률 벡터 (list, np.array, pd.Series)
        maxlags: Newey-West 보정에 사용할 최대 시차
    출력:
        (평균 수익률, NW 표준오차, NW t-통계량)
    """
    returns = np.asarray(returns)
    T = len(returns)
    X = np.ones((T, 1))  # 상수항만 포함 (평균 추정)
    
    model = sm.OLS(returns, X).fit(cov_type='HAC', cov_kwds={'maxlags': maxlags})
    nw_cov = cov_hac(model, nlags=maxlags)
    # nw_se = np.sqrt(nw_cov[0, 0])
    # t_stat = model.params[0] / nw_se
    
    return model.params[0], model.bse[0], model.tvalues[0]


In [5]:
from scipy.stats import t


In [6]:
return_average

,risk_parity,min_var,max_sharpe,paa,equal_strategy,equal_asset_weight
2019-01-30,0.055960,0.031233,-0.045906,-0.023335,0.004488,0.054332
2019-02-28,0.017790,0.007447,-0.022495,-0.016946,-0.003551,0.016301
2019-03-28,-0.006875,-0.012137,-0.013190,-0.012795,-0.011249,-0.004392
2019-04-26,0.022650,-0.002901,0.019038,0.021245,0.015008,0.024158
2019-05-24,-0.025741,-0.008459,-0.030936,-0.034561,-0.024924,-0.027220
...,...,...,...,...,...,...
2024-09-20,0.034817,0.049867,0.025878,0.001493,0.028014,0.034407
2024-10-18,0.003926,-0.001231,0.000756,0.016962,0.005103,0.002778
2024-11-15,-0.044229,-0.033668,-0.023999,-0.020194,-0.030522,-0.020486
2024-12-16,0.020330,0.032332,0.062063,0.078830,0.048389,0.036143


In [16]:
return_average.columns

Index(['risk_parity', 'min_var', 'max_sharpe', 'paa', 'equal_strategy',
       'equal_asset_weight'],
      dtype='object')

In [16]:
bottom_25.describe()

count    19.000000
mean     -0.046634
std       0.035031
min      -0.161342
25%      -0.056741
50%      -0.039852
75%      -0.024435
max      -0.013269
Name: risk_parity, dtype: float64

In [17]:
for col in return_average.columns:
    q1 = return_average[col].quantile(0.25)
    bottom_25 = return_average[return_average[col]<=q1][col]
    print(col)
    desc = bottom_25.describe()
    std_error = desc["std"] / (desc["count"] ** 0.5)

    print(std_error)
    print("skewness:", bottom_25.skew()
            , "kurtosis:", bottom_25.kurtosis())
    print(bottom_25.describe())


risk_parity
0.008036634673393448
skewness: -2.194502809905125 kurtosis: 5.834603577002197
count    19.000000
mean     -0.046634
std       0.035031
min      -0.161342
25%      -0.056741
50%      -0.039852
75%      -0.024435
max      -0.013269
Name: risk_parity, dtype: float64
min_var
0.004805686277502833
skewness: -1.5275341264168882 kurtosis: 1.9544224589765444
count    19.000000
mean     -0.030591
std       0.020948
min      -0.087862
25%      -0.033167
50%      -0.023852
75%      -0.016749
max      -0.008664
Name: min_var, dtype: float64
max_sharpe
0.00839511955102183
skewness: -1.734801441719185 kurtosis: 2.305965969962298
count    19.000000
mean     -0.054128
std       0.036593
min      -0.144492
25%      -0.060663
50%      -0.040911
75%      -0.032261
max      -0.022495
Name: max_sharpe, dtype: float64
paa
0.009595776667162421
skewness: -1.003170912718087 kurtosis: -0.2050044907058819
count    19.000000
mean     -0.073140
std       0.041827
min      -0.156020
25%      -0.093282
50

In [ ]:

    
    # 계산 실행
    mean_return, nw_se, nw_tstat = newey_west_tstat(return_average[col], maxlags=3)



    # 단측 검정 (우측): P(T > t)
    p_value = 1 - t.cdf(nw_tstat, df=len(return_average))
    print(f"Column: {col}")
    print(f"p-value = {p_value:.6f}")
    print(f"Mean Return = {mean_return:.6f}, NW SE = {nw_se:.6f}, NW t-stat = {nw_tstat:.6f}")
    imp.loc[col, "mean_return"] = mean_return
    imp.loc[col, "nw_se"] = nw_se
    imp.loc[col, "nw_tstat"] = nw_tstat
    imp.loc[col, "p_value"] = p_value


In [18]:
imp = pd.DataFrame(index = return_average.columns)

for col in return_average.columns:
    
    # 계산 실행
    mean_return, nw_se, nw_tstat = newey_west_tstat(return_average[col], maxlags=3)



    # 단측 검정 (우측): P(T > t)
    p_value = 1 - t.cdf(nw_tstat, df=len(return_average))
    print(f"Column: {col}")
    print(f"p-value = {p_value:.6f}")
    print(f"Mean Return = {mean_return:.6f}, NW SE = {nw_se:.6f}, NW t-stat = {nw_tstat:.6f}")
    imp.loc[col, "mean_return"] = mean_return
    imp.loc[col, "nw_se"] = nw_se
    imp.loc[col, "nw_tstat"] = nw_tstat
    imp.loc[col, "p_value"] = p_value


Column: risk_parity
p-value = 0.106950
Mean Return = 0.006079, NW SE = 0.004850, NW t-stat = 1.253409
Column: min_var
p-value = 0.015981
Mean Return = 0.008146, NW SE = 0.003728, NW t-stat = 2.185118
Column: max_sharpe
p-value = 0.122460
Mean Return = 0.008445, NW SE = 0.007207, NW t-stat = 1.171847
Column: paa
p-value = 0.307280
Mean Return = 0.003935, NW SE = 0.007782, NW t-stat = 0.505658
Column: equal_strategy
p-value = 0.092324
Mean Return = 0.006651, NW SE = 0.004968, NW t-stat = 1.338731
Column: equal_asset_weight
p-value = 0.107646
Mean Return = 0.006390, NW SE = 0.005113, NW t-stat = 1.249568


In [20]:
imp.T

,risk_parity,min_var,max_sharpe,paa,equal_strategy,equal_asset_weight
mean_return,0.006079,0.008146,0.008445,0.003935,0.006651,0.006390
nw_se,0.004850,0.003728,0.007207,0.007782,0.004968,0.005113
nw_tstat,1.253409,2.185118,1.171847,0.505658,1.338731,1.249568
p_value,0.106950,0.015981,0.122460,0.307280,0.092324,0.107646


In [31]:
nw_tstat

np.float64(1.2891250406400871)

In [25]:
return_average

,date,return
0,2019-01-30,0.001730
1,2019-02-28,0.027784
2,2019-03-28,-0.006986
3,2019-04-26,0.019037
4,2019-05-24,-0.033522
...,...,...
71,2024-09-20,0.009221
72,2024-10-18,0.022761
73,2024-11-15,-0.013343
74,2024-12-16,0.106342


In [26]:
np.quantile(return_average['return'], 0.05)

np.float64(-0.05455205575)